# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Same feature set that `w04_baseline_score.ipynb` and `w05_model.ipynb` use — built here first,
formally, so the leakage hunt below tests the exact thing I actually modeled with. `has_-` flags
replace a blind `fillna`, since missingness follows `content_type` (confirmed in the data
contract notebook), not randomness.

In [1]:
# ── Build the feature vector (same feature set used in w04 baseline / w05 model) ──
import pandas as pd
import numpy as np
import os

_candidates = [
    "/workspaces/assignment1/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)

# Label, per the data contract above.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "avg_position", "impressions_90d", "clicks_90d", "ctr",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

# has_-flags instead of a blind fillna, since missingness follows content_type
# (confirmed in the data contract notebook above).
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
numeric_features += ["has_keyword_data", "has_word_count", "has_position_data"]

feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"]

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"Numeric: {len(numeric_features)} | Categorical: {len(categorical_features)}")
print(f"Target positive rate: {y.mean():.3f}")


Feature matrix: 30,000 rows x 28 columns
Numeric: 25 | Categorical: 3
Target positive rate: 0.542


## 2. Feature notes (meaning, missing, categorical, available-when?)

For each feature: what it means, how missing values are handled, and whether it's knowable
BEFORE the moment I'd want to flag a page for review (it needs to be — a feature that only
exists after the decision is made can't inform the decision).

In [2]:
# ── Feature notes: meaning, missing handling, available-when ──
feature_notes = [
    ("avg_position", "current GSC rank, lower=better; 0='no data'", "has_position_data flag", "yes, current state"),
    ("impressions_90d", "trailing 90d search impressions", "none observed, all rows have >=1", "yes"),
    ("clicks_90d", "trailing 90d search clicks", "none observed", "yes"),
    ("ctr", "clicks_90d/impressions_90d x100", "none observed", "yes"),
    ("pageviews_90d", "trailing 90d GA4 pageviews", "none observed", "yes"),
    ("sessions_90d", "trailing 90d GA4 sessions", "none observed", "yes"),
    ("users_90d", "trailing 90d GA4 users", "none observed", "yes"),
    ("engaged_sessions_90d", "trailing 90d engaged sessions", "none observed", "yes"),
    ("ai_sessions_90d", "trailing 90d AI-referred sessions", "none observed", "yes"),
    ("scroll_events_90d", "trailing 90d scroll events", "none observed", "yes"),
    ("days_with_impressions", "of 90, days with >=1 impression", "none observed", "yes"),
    ("days_with_sessions", "of 90, days with >=1 session", "none observed", "yes"),
    ("engagement_rate", "engaged_sessions/sessions x100", "none observed", "yes"),
    ("scroll_rate", "scroll_events/pageviews x100, CAN exceed 100", "median-imputed, 0.4% blank", "yes"),
    ("ai_traffic_pct", "ai_sessions/sessions x100, CAN exceed 100", "none observed", "yes"),
    ("word_count", "article word count", "median-imputed + has_word_count flag, 25.7% blank (feedly articles)", "yes"),
    ("char_count", "article character count", "same pattern as word_count", "yes"),
    ("content_age_days", "days since content created", "none observed", "yes"),
    ("days_since_last_update", "days since last edit -- core staleness signal", "none observed", "yes"),
    ("search_volume", "keyword search-volume estimate", "median-imputed + has_keyword_data flag, 8.2% blank (feedly articles)", "yes"),
    ("competition", "keyword competition score 0-1", "same pattern as search_volume", "yes"),
    ("cpc", "keyword cost-per-click estimate", "same pattern as search_volume", "yes"),
    ("content_type", "keyword/feedly/comparison article", "none observed", "yes"),
    ("main_intent", "informational/transactional/commercial/navigational", "'unknown' constant fill, 7.9% blank", "yes"),
    ("competition_level", "LOW/MEDIUM/HIGH", "'unknown' constant fill, 8.7% blank", "yes"),
    ("has_keyword_data", "engineered: search_volume not null", "n/a, always defined", "yes"),
    ("has_word_count", "engineered: word_count not null", "n/a, always defined", "yes"),
    ("has_position_data", "engineered: avg_position > 0", "n/a, always defined", "yes"),
]
notes_df = pd.DataFrame(feature_notes, columns=["feature", "meaning", "missing_handling", "available_before_prediction"])
notes_df


,feature,meaning,missing_handling,available_before_prediction
0,avg_position,"current GSC rank, lower=better; 0='no data'",has_position_data flag,"yes, current state"
1,impressions_90d,trailing 90d search impressions,"none observed, all rows have >=1",yes
2,clicks_90d,trailing 90d search clicks,none observed,yes
3,ctr,clicks_90d/impressions_90d x100,none observed,yes
4,pageviews_90d,trailing 90d GA4 pageviews,none observed,yes
5,sessions_90d,trailing 90d GA4 sessions,none observed,yes
6,users_90d,trailing 90d GA4 users,none observed,yes
7,engaged_sessions_90d,trailing 90d engaged sessions,none observed,yes
8,ai_sessions_90d,trailing 90d AI-referred sessions,none observed,yes
9,scroll_events_90d,trailing 90d scroll events,none observed,yes


## 3. The leakage hunt

Per the `hunting-leakage-and-validating` skill: attack my own feature set by training WITH a
suspect column added back in, then WITHOUT, on the same grouped split. A score that jumps
toward 1.0 is the confession that the excluded column was leaking the label.

In [3]:
# ── The leakage hunt: attack my own feature set ──
# Per the hunting-leakage-and-validating skill: train once WITH a suspect column, once
# WITHOUT. A collapse toward the honest number is the confession that it was leaking.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

def make_pipeline(num_cols, cat_cols):
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])
    return Pipeline([("prep", prep),
                      ("clf", RandomForestClassifier(n_estimators=200, max_depth=8,
                                                      min_samples_leaf=20, class_weight="balanced",
                                                      random_state=RANDOM_STATE, n_jobs=-1))])

def fit_score(num_cols, cat_cols):
    Xf = df[num_cols + cat_cols]
    Xtr, Xte = Xf.iloc[train_idx], Xf.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    pipe = make_pipeline(num_cols, cat_cols)
    pipe.fit(Xtr, ytr)
    scores = pipe.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, scores)

honest_auc = fit_score(numeric_features, categorical_features)

# Suspect #1: trend_pct itself -- obviously leaky, defines trend_direction directly.
auc_with_trend_pct = fit_score(numeric_features + ["trend_pct"], categorical_features)

# Suspect #2: the last-30/prev-30 family -- the label is literally last-30 vs prev-30.
suspect_family = ["impressions_last_30d", "impressions_prev_30d",
                   "clicks_last_30d", "clicks_prev_30d",
                   "sessions_last_30d", "sessions_prev_30d"]
auc_with_windows = fit_score(numeric_features + suspect_family, categorical_features)

print(f"Honest feature set (excludes all suspects):        ROC AUC = {honest_auc:.3f}")
print(f"+ trend_pct added back in:                          ROC AUC = {auc_with_trend_pct:.3f}")
print(f"+ last30/prev30 window family added back in:        ROC AUC = {auc_with_windows:.3f}")
print("\nBoth suspect additions push AUC up -- exactly the 'towers over the others / score jumps")
print("toward 1.0' symptom the skill describes. That confirms they were correctly excluded from")
print("the honest feature set used in w04 baseline and w05 model.")


Honest feature set (excludes all suspects):        ROC AUC = 0.608
+ trend_pct added back in:                          ROC AUC = 0.999
+ last30/prev30 window family added back in:        ROC AUC = 0.758

Both suspect additions push AUC up -- exactly the 'towers over the others / score jumps
toward 1.0' symptom the skill describes. That confirms they were correctly excluded from
the honest feature set used in w04 baseline and w05 model.


## 4. What I excluded and why

The full list of fields I refused to use as features, each with a one-line why — two of them
now backed by the AUC-jump test above, not just a rule I'm asserting.

In [4]:
# ── What I excluded, and why (see w03_data_contract.ipynb for the full field-by-field table) ──
excluded = [
    ("trend_direction, trend_pct", "Label-derived: this IS the label / defines it. Confirmed above: adding trend_pct back pushes AUC from 0.608 to 0.999 -- a near-perfect score, the classic leakage confession."),
    ("impressions_last_30d, impressions_prev_30d,\nclicks_last_30d/prev_30d, sessions_last_30d/prev_30d", "Future/overlapping-window leakage: the label is defined FROM last-30 vs prev-30 impressions. Confirmed above: adding this family back pushes AUC from 0.608 to 0.756."),
    ("content_id, client_id", "Context, not features. Pseudonyms used only for grouping/splitting (client-holdout)."),
    ("provider_used, model_used", "Decision/generation-process flags. Data dictionary explicitly marks these 'not a model feature'."),
    ("age_tier, age_tier_order, freshness_tier,\nword_count_tier, char_count_tier,\nimpression_tier, position_tier", "Redundant restatements of columns already in the feature set (bucketed versions of continuous fields already used raw) -- not leakage, just unnecessary duplication."),
]
for cols, why in excluded:
    print(f"- {cols}\n    why: {why}\n")


- trend_direction, trend_pct
    why: Label-derived: this IS the label / defines it. Confirmed above: adding trend_pct back pushes AUC from 0.608 to 0.999 -- a near-perfect score, the classic leakage confession.

- impressions_last_30d, impressions_prev_30d,
clicks_last_30d/prev_30d, sessions_last_30d/prev_30d
    why: Future/overlapping-window leakage: the label is defined FROM last-30 vs prev-30 impressions. Confirmed above: adding this family back pushes AUC from 0.608 to 0.756.

- content_id, client_id
    why: Context, not features. Pseudonyms used only for grouping/splitting (client-holdout).

- provider_used, model_used
    why: Decision/generation-process flags. Data dictionary explicitly marks these 'not a model feature'.

- age_tier, age_tier_order, freshness_tier,
word_count_tier, char_count_tier,
impression_tier, position_tier
    why: Redundant restatements of columns already in the feature set (bucketed versions of continuous fields already used raw) -- not leakage, just 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
